# Build the background-flow cache

Run this notebook once before the background-flow analyses. It uses the current Jupyter kernel, avoiding differences between the notebook environment and the shell Python.

The builder is restartable: completed monthly-file partitions are skipped if the notebook is interrupted and rerun.

In [1]:
# Main user settings
N_WORKERS = 6              # Start with 4; more workers may become I/O-limited
CLIM_WINDOW_DAYS = 91      # Centred: 45 days before through 45 days after

In [2]:
from pathlib import Path
import sys

HERE = Path.cwd()
if not (HERE / 'background_flow_tools.py').exists():
    HERE = Path('seacofs_eddy_tilt_analysis/beta_effect_background_flow').resolve()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, build_background_cache

print('Python:', sys.version.split()[0])
print('Workers:', N_WORKERS)
if sys.version_info < (3, 9):
    raise RuntimeError('Select a Jupyter kernel running Python 3.9 or newer.')

Python: 3.10.8
Workers: 6


## Load and select eddies

The cache is built for all eddy observations. The topographic PV-gradient calculation retains `(w + f)`, but planetary/topographic dominance is deliberately filtered only in the analysis notebook. This makes the expensive cache reusable for alternative thresholds and subsets.

In [3]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
eddies = tilt.add_pv_gradient_terms(eddies, grid)
selected = eddies.copy()
print(f"Selected {len(selected):,} observations from {selected['Eddy'].nunique():,} eddies")

Selected 127,426 observations from 2,982 eddies


## Build or resume the cache

This is the long-running cell on the first run. It reads each archive file once and accumulates monthly and full-archive climatological fields. Each eddy-day is assigned a centred 91-day moving seasonal climatology by weighting the monthly fields according to how many days from each month fall in its 45-day-before/45-day-after window. There is no annulus estimate. On rerun, completed file partitions are reused.

In [4]:
config = BackgroundConfig(climatology_window_days=CLIM_WINDOW_DAYS)
background = build_background_cache(
    selected, grid, config=config, workers=N_WORKERS
)
print('Cache complete:', config.background_table_path)
print(f"Rows: {len(background):,}; eddies: {background['Eddy'].nunique():,}")

Required levels by depth: {200: 30, 500: 30}; reading 30/30 surface sigma levels.
All sigma levels are required because sampled shallow columns lie entirely above the depth limit.


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:   12.5s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   17.3s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   35.0s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   48.2s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:  1.2min
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:  1.4min
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:  1.9min
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  2.3min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  2.8min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  3.2min
[Parallel(n_jobs=6)]: Done 101 tasks      | elapsed:  3.9min
[Parallel(n_jobs=6)]: Done 116 tasks      | elapsed:  4.5min
[Parallel(n_jobs=6)]: Done 133 tasks      | elapsed:  5.2min
[Parallel(n_jobs=6)]: Done 150 tasks      | elapsed:  6.0min
[Parallel(n_jobs=6)]: Done 169 tasks      | elapsed:  6.8min
[Parallel(

Cache complete: /srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/background_flow_cache_all_eddies_v4/eddy_day_background.parquet
Rows: 127,426; eddies: 2,982


When the final cell finishes, open and run `background_relative_beta_effect.ipynb`. If this notebook is interrupted, rerun it with the same settings; completed monthly files will be reused.